# Marco 3 — Experimentação comparativa (versão revisada)

**Grupo G3** — TP1 de Tópicos Especiais em Sistemas de Informação

Objetivos (Semana 3 do cronograma do TP1):
1. Extração das **3 famílias de descritores** (textura GLCM, forma/contorno, gradiente/bordas)
2. Grade **descritor × modelo** com validação cruzada aninhada (partição por paciente **congelada no Marco 2**)
3. Estudo de ablação (pré-processamento **e** agregação de cortes)
4. Tabelas e figuras finais + preparação para a análise de erro (Marco 4)

**O que mudou nesta versão (sem alterar o experimento congelado):**
- **Cache de features:** se `features_todas_familias_g3.csv` já existir, ele é reaproveitado — não é preciso reler os DICOM nem rodar o Marco 2 de novo. Só a ablação de agregação de cortes exige DICOM (uma vez; o resultado é salvo em CSV).
- **Caminhos** sem usuário/conta específica (busca automática + variável de ambiente `RSNA_DATA_DIR`).
- **Métricas completas do §4.4:** AUC-ROC, **AUC-PR**, sensibilidade, especificidade, **F1**, acurácia balanceada; AUC por dobra *e* agrupado (fora-da-dobra).
- **Baseline trivial com probabilidades** (`strategy="prior"`), permitindo AUC/AUC-PR comparáveis.
- **Estimativa sem viés de seleção:** o par descritor×modelo é escolhido *dentro* do treino de cada dobra externa.
- **Incerteza:** IC 95% por bootstrap e teste de permutação.
- **Ablação de agregação de cortes** (corte central × pooling de 3/5/9 cortes) — decisão exigida pelo §4.2.
- **Redução de dimensionalidade** (SelectKBest) para o conjunto combinado.
- **Preparação da análise de erro:** predições com ID do paciente, plano de aquisição, exemplos de erros.
- **Figuras corrigidas** (matriz de confusão, ROC, barras).

## 1. Setup e configuração

In [ ]:
!pip install -q pydicom scikit-image opencv-python-headless

import os
import glob
import json
import platform
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import pydicom
import sklearn
import skimage
import scipy
from scipy.stats import chi2_contingency
from skimage.transform import resize
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu, sobel_h, sobel_v
from skimage.measure import label, regionprops
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_recall_curve, f1_score, balanced_accuracy_score,
                             accuracy_score, confusion_matrix)

SEED = 42
np.random.seed(SEED)

# ---- Configuração do experimento -------------------------------------------------
FEATURE_MODALITY = "T1wCE"   # confirmado por experimentação no Marco 2
USE_CACHE = True             # reaproveita CSVs de features já gerados (evita reler DICOM)
N_BOOT = 2000                # reamostragens do bootstrap
N_PERM = 200                 # permutações do teste de permutação
PERM_SCOPE = "melhor"        # "melhor" (rápido) ou "todas" (as 12 combinações; lento)
RUN_AGG_ABLATION = True      # ablação de agregação de cortes (exige DICOM na 1ª vez)
RUN_ERROR_PLOTS = True       # figura com exemplos de erros (exige DICOM)
MULTISLICE_OFFSETS = list(range(-4, 5))   # cortes centrais: -4..+4 (9 cortes)

OUT_DIR = "outputs_marco3_g3"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- Localização de dados e arquivos (sem caminhos pessoais) -----------------------
def find_dir(cands):
    for c in cands:
        if c and os.path.isdir(c):
            return c
    return None

DATA_DIR = find_dir([
    os.environ.get("RSNA_DATA_DIR"),
    "/kaggle/input/competitions/rsna-miccai-brain-tumor-radiogenomic-classification",
    "/kaggle/input/rsna-miccai-brain-tumor-radiogenomic-classification",
    "data/rsna-miccai-brain-tumor-radiogenomic-classification",
])
TRAIN_DIR = os.path.join(DATA_DIR, "train") if DATA_DIR else None
print("DATA_DIR:", DATA_DIR if DATA_DIR else "não encontrado (só será possível usar features em cache)")

LOCAL_DIRS = [".", "outputs_eda_g3", "outputs_marco2_g3", "outputs_marco3_g3",
              "outputs/eda", "outputs/marco2", "outputs/marco3", "data"]
KAGGLE_PATTERNS = ["/kaggle/input/*/{n}", "/kaggle/input/*/*/{n}", "/kaggle/input/*/*/*/{n}",
                   "/kaggle/input/notebooks/*/*/*/{n}", "/kaggle/input/*/outputs*/{n}"]

def find_file(name, required=True):
    for d in LOCAL_DIRS:
        p = os.path.join(d, name)
        if os.path.isfile(p):
            return p
    for pat in KAGGLE_PATTERNS:
        hits = sorted(glob.glob(pat.format(n=name)))
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(
            f"'{name}' não encontrado. Rode o notebook anterior (Marco 1/2) ou adicione o "
            f"output dele como Input no Kaggle / copie o arquivo para uma das pastas: {LOCAL_DIRS}")
    return None

def load_ids_csv(name):
    d = pd.read_csv(find_file(name))
    d["patient_id_str"] = d["patient_id_str"].astype(str).str.zfill(5)
    return d

# Amostra (Marco 1) e partição congelada (Marco 2) — fontes únicas de verdade
amostra_df = load_ids_csv("amostra_g3.csv")
particao_df = load_ids_csv("particao_pacientes_congelada.csv")
df = amostra_df.merge(particao_df[["patient_id_str", "fold"]], on="patient_id_str", how="inner")
assert len(df) == len(amostra_df), "Partição do Marco 2 não cobre toda a amostra — conferir arquivos"
assert df.groupby("patient_id_str")["fold"].nunique().max() == 1
print(f"Pacientes: {len(df)} | dobras: {sorted(df['fold'].unique())}")
df.groupby(["fold", "MGMT_value"]).size().unstack()

## 2. Leitura e pré-processamento (mesmo pipeline do Marco 2)

Leitura *orientation-aware* (ordena por posição física), normalização por percentil (1–99%),
recorte da região não-nula e redimensionamento para 128×128.

In [ ]:
def read_series(patient_id, modality, data_dir=None):
    data_dir = data_dir or TRAIN_DIR
    series_dir = os.path.join(data_dir, patient_id, modality)
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    slices = [pydicom.dcmread(f) for f in files]
    iop = getattr(slices[0], "ImageOrientationPatient", None) if slices else None
    if iop is not None:
        row_vec = np.array(iop[0:3], dtype=float)
        col_vec = np.array(iop[3:6], dtype=float)
        normal = np.cross(row_vec, col_vec)
        try:
            slices.sort(key=lambda ds: float(np.dot(np.array(ds.ImagePositionPatient, dtype=float), normal)))
        except Exception:
            slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))
    else:
        slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))
    return slices


def preprocess_slice(img, target_size=128, normalize=True, crop=True):
    img = img.astype(np.float32)
    if normalize:
        p1, p99 = np.percentile(img, [1, 99])
        img = np.clip(img, p1, p99)
        if p99 > p1:
            img = (img - p1) / (p99 - p1)
        else:
            img = np.zeros_like(img)
    else:
        # apenas min-max simples, sem clipping de percentil (usado na ablação)
        mn, mx = img.min(), img.max()
        img = (img - mn) / (mx - mn) if mx > mn else np.zeros_like(img)

    if crop:
        mask = img > 0.02
        if mask.any():
            rows = np.any(mask, axis=1)
            cols = np.any(mask, axis=0)
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            img = img[rmin:rmax+1, cmin:cmax+1]

    return resize(img, (target_size, target_size), anti_aliasing=True)


def get_middle_slice(patient_id, modality, **preprocess_kwargs):
    series = read_series(patient_id, modality)
    ds = series[len(series) // 2]
    return preprocess_slice(ds.pixel_array, **preprocess_kwargs)


def get_slices_by_offset(patient_id, modality, offsets, **preprocess_kwargs):
    """Lê a série UMA vez e devolve {offset: imagem pré-processada}, com offset 0 = corte do meio
    (mesmo critério de get_middle_slice). Índices fora da série são truncados nas extremidades."""
    series = read_series(patient_id, modality)
    n, mid = len(series), len(series) // 2
    out = {}
    for off in offsets:
        idx = min(max(mid + off, 0), n - 1)
        out[off] = preprocess_slice(series[idx].pixel_array, **preprocess_kwargs)
    return out

## 3. Famílias de descritores

**Família 1 — Textura (GLCM/Haralick)**: T1wCE, distância 1, 32 níveis de cinza (Marco 2).
**Família 2 — Forma/contorno**: momentos de Hu + propriedades de região.
*Limitação documentada:* o dataset da Task 2 não traz máscara do tumor; a forma é a da maior região
de tecido após Otsu (aproxima o contorno do cérebro no corte, não do tumor).
**Família 3 — Gradiente/bordas**: histograma de orientações (Sobel, 9 bins, ponderado pela magnitude) + densidade de bordas (Canny).

In [ ]:
GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
N_GRAY_LEVELS = 32
GLCM_DISTANCE = 1

def extract_glcm_features(img01):
    img_uint = (img01 * (N_GRAY_LEVELS - 1)).astype(np.uint8)
    glcm = graycomatrix(img_uint, distances=[GLCM_DISTANCE], angles=GLCM_ANGLES,
                        levels=N_GRAY_LEVELS, symmetric=True, normed=True)
    feats = {}
    for prop in GLCM_PROPS:
        values = graycoprops(glcm, prop)[0]
        feats[f"glcm_{prop}_mean"] = np.mean(values)
        feats[f"glcm_{prop}_std"] = np.std(values)
    return feats


def extract_shape_features(img01):
    try:
        thresh = threshold_otsu(img01)
    except ValueError:
        thresh = 0.5
    binary = (img01 > thresh).astype(np.uint8)

    labeled = label(binary)
    props = regionprops(labeled)
    if not props:
        return {k: 0.0 for k in ["shape_area_frac", "shape_eccentricity", "shape_solidity",
                                 "shape_perimeter_frac", "shape_extent"] + [f"shape_hu{i+1}" for i in range(7)]}

    largest = max(props, key=lambda p: p.area)
    total_px = img01.shape[0] * img01.shape[1]
    feats = {
        "shape_area_frac": largest.area / total_px,
        "shape_eccentricity": largest.eccentricity,
        "shape_solidity": largest.solidity,
        "shape_perimeter_frac": largest.perimeter / (img01.shape[0] + img01.shape[1]),
        "shape_extent": largest.extent,
    }
    mask_largest = (labeled == largest.label).astype(np.uint8)
    hu = cv2.HuMoments(cv2.moments(mask_largest)).flatten()
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)
    for i, val in enumerate(hu_log):
        feats[f"shape_hu{i+1}"] = val
    return feats


N_ORIENTATION_BINS = 9

def extract_gradient_features(img01):
    img_uint8 = (img01 * 255).astype(np.uint8)
    gx = sobel_v(img01)
    gy = sobel_h(img01)
    magnitude = np.sqrt(gx**2 + gy**2)
    orientation_deg = (np.degrees(np.arctan2(gy, gx)) + 180) % 180
    hist, _ = np.histogram(orientation_deg, bins=N_ORIENTATION_BINS, range=(0, 180), weights=magnitude)
    hist = hist / (hist.sum() + 1e-10)

    feats = {f"grad_orient_bin{i}": v for i, v in enumerate(hist)}
    feats["grad_magnitude_mean"] = magnitude.mean()
    feats["grad_magnitude_std"] = magnitude.std()
    edges = cv2.Canny(img_uint8, threshold1=50, threshold2=150)
    feats["grad_edge_density"] = (edges > 0).mean()
    return feats


def extract_all_families(img01):
    feats = {}
    feats.update(extract_glcm_features(img01))
    feats.update(extract_shape_features(img01))
    feats.update(extract_gradient_features(img01))
    return feats

## 4. Extração das features (com cache)

`load_or_build` procura o CSV em cache; se não existir (ou estiver incompleto), gera a partir dos DICOM e salva
em `outputs_marco3_g3/`. Assim, **não é necessário reler as imagens** se o CSV de features já existe.

In [ ]:
def load_or_build(name, builder, required_ids=None):
    """Retorna DataFrame de features (cache -> DICOM). None se não houver como obter."""
    path_out = os.path.join(OUT_DIR, name)
    found = find_file(name, required=False) if USE_CACHE else None
    if found:
        d = pd.read_csv(found)
        d["patient_id_str"] = d["patient_id_str"].astype(str).str.zfill(5)
        if required_ids is None or set(required_ids) <= set(d["patient_id_str"]):
            print(f"[cache] {name} <- {found}")
            if os.path.abspath(found) != os.path.abspath(path_out):
                d.to_csv(path_out, index=False)
            return d
        print(f"[cache incompleto] {name} — regenerando a partir dos DICOM")
    if TRAIN_DIR is None or not os.path.isdir(TRAIN_DIR):
        print(f"[aviso] sem cache e sem DATA_DIR: não foi possível gerar '{name}'.")
        return None
    d = builder()
    d.to_csv(path_out, index=False)
    print(f"[gerado] {path_out} ({len(d)} linhas)")
    return d


def build_central_features(**preprocess_kwargs):
    rows, failed = [], []
    for pid in df["patient_id_str"]:
        try:
            img = get_middle_slice(pid, FEATURE_MODALITY, **preprocess_kwargs)
            feats = {"patient_id_str": pid}
            feats.update(extract_all_families(img))
            rows.append(feats)
        except Exception as e:
            failed.append((pid, str(e)))
    if failed:
        print("Falhas:", failed)
    return pd.DataFrame(rows)


PATIENT_IDS = df["patient_id_str"].tolist()
all_features_df = load_or_build("features_todas_familias_g3.csv", build_central_features, PATIENT_IDS)
assert all_features_df is not None, "Sem features e sem DICOM — configure RSNA_DATA_DIR ou forneça o CSV."
print(f"Features para {len(all_features_df)} de {len(df)} pacientes")

GLCM_COLS = [c for c in all_features_df.columns if c.startswith("glcm_")]
SHAPE_COLS = [c for c in all_features_df.columns if c.startswith("shape_")]
GRAD_COLS = [c for c in all_features_df.columns if c.startswith("grad_")]
COMBINED_COLS = GLCM_COLS + SHAPE_COLS + GRAD_COLS
print(f"n features — GLCM: {len(GLCM_COLS)} | Forma: {len(SHAPE_COLS)} | "
      f"Gradiente: {len(GRAD_COLS)} | Combinado: {len(COMBINED_COLS)}")
all_features_df.head()

## 5. Métricas e protocolo de validação (§4.3–4.4 do TP1)

- **Conjuntos de descritores:** GLCM, Forma, Gradiente, Combinado.
- **Modelos (≥3):** regressão logística (L2), Random Forest, SVM (RBF).
- **Validação:** 5 dobras externas *congeladas* (por paciente); hiperparâmetros por `GridSearchCV` (3 dobras internas)
  apenas no treino de cada dobra externa. Padronização dentro do `Pipeline`.
- **Métricas por dobra:** AUC-ROC, AUC-PR, acurácia balanceada, sensibilidade, especificidade, F1 e acurácia (acurácia isolada não é aceita).
- **Agregações:** média ± desvio das dobras **e** AUC agrupado (fora-da-dobra, mais conservador — mistura calibrações de modelos diferentes).
- Para cada dobra também é guardado o **score interno** do melhor modelo, usado na seleção sem viés (Seção 8).

In [ ]:
DESCRIPTOR_SETS = {"GLCM": GLCM_COLS, "Forma": SHAPE_COLS, "Gradiente": GRAD_COLS, "Combinado": COMBINED_COLS}

MODEL_GRID = {
    "LogReg": (LogisticRegression(max_iter=1000, random_state=SEED),
               {"clf__C": [0.1, 1.0, 10.0]}),
    "RandomForest": (RandomForestClassifier(random_state=SEED),
                     {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 5]}),
    "SVM": (SVC(kernel="rbf", probability=True, random_state=SEED),
            {"clf__C": [0.1, 1.0, 10.0], "clf__gamma": ["scale", "auto"]}),
}

METRICS = ["auc", "auc_pr", "balanced_accuracy", "sensibilidade", "especificidade", "f1", "accuracy"]


def fold_metrics(y_val, preds, proba):
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    both = len(np.unique(y_val)) > 1
    return {
        "accuracy": accuracy_score(y_val, preds),
        "balanced_accuracy": balanced_accuracy_score(y_val, preds),
        "sensibilidade": tp / (tp + fn) if (tp + fn) > 0 else np.nan,
        "especificidade": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "f1": f1_score(y_val, preds, zero_division=0),
        "auc": roc_auc_score(y_val, proba) if both else np.nan,
        "auc_pr": average_precision_score(y_val, proba) if both else np.nan,
    }


def evaluate_nested_cv(merged_df, feature_cols, base_model, param_grid, inner_folds=3,
                       selector=None, n_jobs=-1, y_col="MGMT_value"):
    """CV aninhada: dobras externas = partição congelada; GridSearchCV interno só no treino.
    Retorna (fold_df, oof_df). oof_df guarda as predições fora-da-dobra com o ID do paciente."""
    fold_rows, oof_rows = [], []
    for fold in sorted(merged_df["fold"].unique()):
        train_df = merged_df[merged_df["fold"] != fold]
        val_df = merged_df[merged_df["fold"] == fold]

        steps = [("scaler", StandardScaler())]
        if selector is not None:
            steps.append(("sel", selector))
        steps.append(("clf", base_model))
        grid = GridSearchCV(Pipeline(steps), param_grid, cv=inner_folds, scoring="roc_auc", n_jobs=n_jobs)
        grid.fit(train_df[feature_cols], train_df[y_col])

        best = grid.best_estimator_
        preds = best.predict(val_df[feature_cols])
        proba = best.predict_proba(val_df[feature_cols])[:, 1]
        y_val = val_df[y_col].values

        row = {"fold": fold, **fold_metrics(y_val, preds, proba),
               "score_interno": grid.best_score_, "melhores_hiperparametros": str(grid.best_params_)}
        fold_rows.append(row)
        oof_rows.append(pd.DataFrame({"patient_id_str": val_df["patient_id_str"].values, "fold": fold,
                                      "y_true": y_val, "proba": proba, "pred": preds}))
    return pd.DataFrame(fold_rows), pd.concat(oof_rows, ignore_index=True)


def evaluate_baseline(merged_df):
    """Baseline trivial (classe majoritária), com probabilidades = prior do treino."""
    fold_rows, oof_rows = [], []
    for fold in sorted(merged_df["fold"].unique()):
        train_df = merged_df[merged_df["fold"] != fold]
        val_df = merged_df[merged_df["fold"] == fold]
        clf = DummyClassifier(strategy="prior")
        clf.fit(np.zeros((len(train_df), 1)), train_df["MGMT_value"])
        Xv = np.zeros((len(val_df), 1))
        preds, proba = clf.predict(Xv), clf.predict_proba(Xv)[:, 1]
        y_val = val_df["MGMT_value"].values
        fold_rows.append({"fold": fold, **fold_metrics(y_val, preds, proba)})
        oof_rows.append(pd.DataFrame({"patient_id_str": val_df["patient_id_str"].values, "fold": fold,
                                      "y_true": y_val, "proba": proba, "pred": preds}))
    return pd.DataFrame(fold_rows), pd.concat(oof_rows, ignore_index=True)


def summarise(fold_df, oof_df):
    out = {}
    for m in METRICS:
        out[f"{m}_mean"] = fold_df[m].mean()
        out[f"{m}_std"] = fold_df[m].std()
    out["auc_pooled"] = roc_auc_score(oof_df["y_true"], oof_df["proba"])
    out["auc_pr_pooled"] = average_precision_score(oof_df["y_true"], oof_df["proba"])
    return out


def fmt(m, s):
    return f"{m:.3f} ± {s:.3f}"

## 6. Baseline trivial (classe majoritária) com a mesma partição

In [ ]:
base_fold_df, base_oof_df = evaluate_baseline(df)
base_summary = summarise(base_fold_df, base_oof_df)
base_fold_df.to_csv(os.path.join(OUT_DIR, "baseline_trivial_prior_metricas.csv"), index=False)

print("Baseline trivial (prior) — média ± desvio entre dobras:")
for m in METRICS:
    print(f"  {m:18s} {fmt(base_summary[m + '_mean'], base_summary[m + '_std'])}")
print("\nObs.: a acurácia < 0,5 é artefato de amostra pequena (a classe majoritária do treino tende a ser a\n"
      "minoritária na validação). Compare pela acurácia balanceada e pelo AUC (0,5 por construção).")
base_fold_df

## 7. Grade descritor × modelo (12 combinações)

Mesmos hiperparâmetros e mesma partição do experimento congelado — os AUCs por dobra devem reproduzir os já obtidos.
Além da tabela-resumo, salvamos as métricas **por dobra** e as **predições fora-da-dobra com o ID do paciente**
(insumo da análise de erro).

In [ ]:
grid_results, oof_all, fold_all = [], {}, []

for desc_name, desc_cols in DESCRIPTOR_SETS.items():
    merged = df.merge(all_features_df[["patient_id_str"] + desc_cols], on="patient_id_str", how="inner")
    for model_name, (base_model, param_grid) in MODEL_GRID.items():
        fold_df, oof_df = evaluate_nested_cv(merged, desc_cols, base_model, param_grid)
        oof_all[(desc_name, model_name)] = oof_df
        fold_all.append(fold_df.assign(descritor=desc_name, modelo=model_name))

        summary = {"descritor": desc_name, "modelo": model_name, "n_features": len(desc_cols),
                   **summarise(fold_df, oof_df)}
        grid_results.append(summary)
        print(f"{desc_name:10s} | {model_name:12s} -> AUC {fmt(summary['auc_mean'], summary['auc_std'])} "
              f"| AUC agrupado {summary['auc_pooled']:.3f} | AUC-PR {summary['auc_pr_mean']:.3f} "
              f"| BalAcc {summary['balanced_accuracy_mean']:.3f}")

grid_results_df = pd.DataFrame(grid_results).sort_values("auc_mean", ascending=False).reset_index(drop=True)
fold_all_df = pd.concat(fold_all, ignore_index=True)
oof_all_df = pd.concat([o.assign(descritor=k[0], modelo=k[1]) for k, o in oof_all.items()], ignore_index=True)

grid_results_df.to_csv(os.path.join(OUT_DIR, "grade_comparativa_descritor_modelo.csv"), index=False)
fold_all_df.to_csv(os.path.join(OUT_DIR, "grade_metricas_por_dobra.csv"), index=False)
oof_all_df.to_csv(os.path.join(OUT_DIR, "predicoes_oof_todas_combinacoes.csv"), index=False)
grid_results_df

## 8. Estimativa sem viés de seleção (seleção aninhada do par descritor × modelo)

Escolher a melhor das 12 combinações *olhando as mesmas dobras que serão reportadas* é otimista
("escolher o melhor modelo pelo conjunto de teste" — §10, erro 4 do TP1). Aqui a escolha é refeita **em cada dobra externa
usando apenas o score da validação interna do treino** (`score_interno`); só então se olha o teste daquela dobra.
Essa é a estimativa que deve ser reportada como desempenho final do procedimento "escolher o melhor par".

In [ ]:
rows_sel, oof_sel = [], []
for fold, g in fold_all_df.groupby("fold"):
    escolhido = g.loc[g["score_interno"].idxmax()]
    rows_sel.append(escolhido)
    o = oof_all[(escolhido["descritor"], escolhido["modelo"])]
    oof_sel.append(o[o["fold"] == fold])

sel_df = pd.DataFrame(rows_sel)[["fold", "descritor", "modelo", "score_interno"] + METRICS].reset_index(drop=True)
oof_sel_df = pd.concat(oof_sel, ignore_index=True)
sel_summary = summarise(sel_df, oof_sel_df)
sel_df.to_csv(os.path.join(OUT_DIR, "selecao_aninhada_por_dobra.csv"), index=False)

print("Par escolhido em cada dobra externa (só pelo score interno do treino):")
print(sel_df[["fold", "descritor", "modelo", "score_interno", "auc"]].to_string(index=False))
print(f"\nSeleção aninhada — AUC: {fmt(sel_summary['auc_mean'], sel_summary['auc_std'])} | "
      f"AUC agrupado: {sel_summary['auc_pooled']:.3f} | AUC-PR: {sel_summary['auc_pr_mean']:.3f} | "
      f"BalAcc: {sel_summary['balanced_accuracy_mean']:.3f}")

## 9. Comparação final: baseline trivial × melhor combinação × seleção aninhada

In [ ]:
melhor_combo = grid_results_df.iloc[0]
BEST_KEY = (melhor_combo["descritor"], melhor_combo["modelo"])
best_oof = oof_all[BEST_KEY].copy()
best_desc_cols = DESCRIPTOR_SETS[BEST_KEY[0]]
print(f"Melhor combinação (nas dobras externas): {BEST_KEY[0]} + {BEST_KEY[1]}")

def row_from(summary, nome, pooled_valido=True):
    return {"abordagem": nome,
            "AUC (média±dp)": fmt(summary["auc_mean"], summary["auc_std"]),
            # no baseline, o prior muda de dobra para dobra: o AUC agrupado é artefato (não interpretável)
            "AUC agrupado": round(summary["auc_pooled"], 3) if pooled_valido else "n/d (artefato do prior por dobra)",
            "AUC-PR (média±dp)": fmt(summary["auc_pr_mean"], summary["auc_pr_std"]),
            "Acc. balanceada": fmt(summary["balanced_accuracy_mean"], summary["balanced_accuracy_std"]),
            "Sensibilidade": fmt(summary["sensibilidade_mean"], summary["sensibilidade_std"]),
            "Especificidade": fmt(summary["especificidade_mean"], summary["especificidade_std"]),
            "F1": fmt(summary["f1_mean"], summary["f1_std"]),
            "Acurácia": fmt(summary["accuracy_mean"], summary["accuracy_std"])}

comparativo_df = pd.DataFrame([
    row_from(base_summary, "Baseline trivial (classe majoritária)", pooled_valido=False),
    row_from(melhor_combo.to_dict(), f"Melhor combinação: {BEST_KEY[0]} + {BEST_KEY[1]} (otimista)"),
    row_from(sel_summary, "Seleção aninhada do par (estimativa sem viés de seleção)"),
])
comparativo_df.to_csv(os.path.join(OUT_DIR, "comparativo_baseline_vs_melhor.csv"), index=False)
comparativo_df.T

## 10. Incerteza: IC por bootstrap e teste de permutação

Com 60 pacientes, o AUC tem erro-padrão grande. Reportamos:
- **IC 95% (bootstrap por paciente)** do AUC e AUC-PR agrupados da melhor combinação;
- **Teste de permutação:** embaralha os rótulos, refaz *todo* o procedimento (CV aninhada) e compara com o AUC observado.
  Com `PERM_SCOPE="melhor"` só a melhor combinação é refeita (não corrige a escolha entre as 12; para isso, use `"todas"`).

In [ ]:
def bootstrap_ci(y, p, metric, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(metric(y[idx], p[idx]))
    return np.percentile(vals, [2.5, 97.5])

y_b, p_b = best_oof["y_true"].values, best_oof["proba"].values
auc_pooled_best = roc_auc_score(y_b, p_b)
ap_pooled_best = average_precision_score(y_b, p_b)
ci_auc = bootstrap_ci(y_b, p_b, roc_auc_score)
ci_ap = bootstrap_ci(y_b, p_b, average_precision_score)
print(f"{BEST_KEY[0]} + {BEST_KEY[1]}")
print(f"  AUC agrupado   = {auc_pooled_best:.3f}  IC95% [{ci_auc[0]:.3f}, {ci_auc[1]:.3f}]")
print(f"  AUC-PR agrupado = {ap_pooled_best:.3f}  IC95% [{ci_ap[0]:.3f}, {ci_ap[1]:.3f}]")
if ci_auc[0] <= 0.5:
    print("  -> o IC do AUC inclui 0,5: resultado estatisticamente inconclusivo com n = 60.")

In [ ]:
def permutation_test(combos, n_perm=N_PERM, seed=SEED):
    """Estatística: maior AUC médio por dobra entre `combos` (1 combinação => AUC da melhor)."""
    rng = np.random.default_rng(seed)
    merged_by_desc = {d: df.merge(all_features_df[["patient_id_str"] + DESCRIPTOR_SETS[d]],
                                  on="patient_id_str", how="inner")
                      for d in {c[0] for c in combos}}

    def stat(perm_map):
        best = -np.inf
        for d, m in combos:
            mg = merged_by_desc[d].copy()
            mg["y_perm"] = mg["patient_id_str"].map(perm_map).astype(int)
            base_model, grid = MODEL_GRID[m]
            f_df, _ = evaluate_nested_cv(mg, DESCRIPTOR_SETS[d], base_model, grid,
                                         n_jobs=1, y_col="y_perm")
            best = max(best, f_df["auc"].mean())
        return best

    observed = max(grid_results_df.set_index(["descritor", "modelo"]).loc[c, "auc_mean"] for c in combos)
    ids, y_orig = df["patient_id_str"].values, df["MGMT_value"].values
    null = []
    for _ in range(n_perm):
        null.append(stat(dict(zip(ids, rng.permutation(y_orig)))))
    null = np.array(null)
    p_val = (1 + np.sum(null >= observed)) / (1 + n_perm)
    return observed, null, p_val

combos_perm = [BEST_KEY] if PERM_SCOPE == "melhor" else list(oof_all.keys())
obs_auc, null_auc, p_perm = permutation_test(combos_perm)
print(f"Escopo: {PERM_SCOPE} | permutações: {N_PERM}")
print(f"AUC observado (médio por dobra): {obs_auc:.3f}")
print(f"AUC sob H0: média {null_auc.mean():.3f} | percentil 95 = {np.percentile(null_auc, 95):.3f}")
print(f"p-valor de permutação: {p_perm:.4f}")

plt.figure(figsize=(6, 4))
plt.hist(null_auc, bins=25, color="lightgray", edgecolor="gray")
plt.axvline(obs_auc, color="crimson", label=f"observado = {obs_auc:.3f} (p = {p_perm:.3f})")
plt.xlabel("AUC médio por dobra sob rótulos embaralhados")
plt.ylabel("Frequência")
plt.title("Teste de permutação", fontsize=11)
plt.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "permutacao_auc_nula.png"), dpi=150)
plt.show()

pd.DataFrame([{"combinacao": f"{BEST_KEY[0]} + {BEST_KEY[1]}", "escopo_permutacao": PERM_SCOPE,
               "auc_agrupado": auc_pooled_best, "auc_agrupado_ic95_inf": ci_auc[0], "auc_agrupado_ic95_sup": ci_auc[1],
               "auc_pr_agrupado": ap_pooled_best, "auc_pr_ic95_inf": ci_ap[0], "auc_pr_ic95_sup": ci_ap[1],
               "auc_medio_dobras": obs_auc, "n_permutacoes": N_PERM, "p_valor_permutacao": p_perm}]
             ).to_csv(os.path.join(OUT_DIR, "incerteza_melhor_combinacao.csv"), index=False)

## 11. Figuras

Correções em relação à versão anterior: legenda fora da área das barras, título da ROC/matriz sem corte,
matriz de confusão com escala fixa em 0 (antes, o menor valor virava "branco" e o número sumia), mais curva PR.

In [ ]:
# (a) AUC médio por descritor x modelo, com desvio entre dobras
order = ["GLCM", "Forma", "Gradiente", "Combinado"]
piv = grid_results_df.pivot(index="descritor", columns="modelo", values="auc_mean").reindex(order)
piv_sd = grid_results_df.pivot(index="descritor", columns="modelo", values="auc_std").reindex(order)

fig, ax = plt.subplots(figsize=(9, 5))
piv.plot(kind="bar", ax=ax, yerr=piv_sd, capsize=3, width=0.8)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="acaso (AUC = 0,5)")
ax.set_ylabel("AUC médio (5 dobras) ± desvio")
ax.set_xlabel("Descritor")
ax.set_title("Grade comparativa: descritor × modelo")
ax.set_xticklabels(order, rotation=0)
ax.legend(title="Modelo", loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "grafico_auc_descritor_modelo.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# (b) Curva ROC da melhor combinação: por dobra (finas) + agrupada (grossa)
fig, ax = plt.subplots(figsize=(5.6, 5.2))
for fold, g in best_oof.groupby("fold"):
    if g["y_true"].nunique() > 1:
        f_, t_, _ = roc_curve(g["y_true"], g["proba"])
        ax.plot(f_, t_, color="lightsteelblue", linewidth=1)
fpr, tpr, _ = roc_curve(y_b, p_b)
ax.plot(fpr, tpr, color="C0", linewidth=2.2,
        label=f"agrupado: AUC = {auc_pooled_best:.3f}\nIC95% [{ci_auc[0]:.2f}; {ci_auc[1]:.2f}]")
ax.plot([], [], color="lightsteelblue", label=f"dobras (AUC médio = {melhor_combo['auc_mean']:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="acaso")
ax.set_xlabel("Taxa de falso positivo")
ax.set_ylabel("Taxa de verdadeiro positivo")
ax.set_title(f"ROC — {BEST_KEY[0]} + {BEST_KEY[1]} (fora-da-dobra)", fontsize=11)
ax.legend(loc="lower right", fontsize=8.5)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_melhor_combinacao.png"), dpi=150)
plt.show()

# (c) Curva Precisão-Revocação
prec, rec, _ = precision_recall_curve(y_b, p_b)
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot(rec, prec, color="C1", linewidth=2, label=f"AUC-PR = {ap_pooled_best:.3f}")
ax.axhline(y_b.mean(), linestyle="--", color="gray", label=f"prevalência = {y_b.mean():.2f}")
ax.set_xlabel("Revocação (sensibilidade)")
ax.set_ylabel("Precisão")
ax.set_title(f"PR — {BEST_KEY[0]} + {BEST_KEY[1]}", fontsize=11)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pr_melhor_combinacao.png"), dpi=150)
plt.show()

In [ ]:
# (d) Matriz de confusão (limiar 0,5) com escala fixa e texto legível
cm = confusion_matrix(best_oof["y_true"], best_oof["pred"], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.6, 4.2))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
ax.set_xticks([0, 1]); ax.set_xticklabels(["Não metilado", "Metilado"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Não metilado", "Metilado"])
ax.set_xlabel("Predito"); ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusão\n{BEST_KEY[0]} + {BEST_KEY[1]}", fontsize=11)
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13,
                color="white" if cm[i, j] > cm.max() * 0.55 else "black")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "matriz_confusao_melhor_combinacao.png"), dpi=150)
plt.show()
print(f"VN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  VP={cm[1,1]}")

## 12. Ablação do pré-processamento (§4.1 do TP1)

Referência: melhor combinação da grade. Configurações: pipeline completo × sem normalização por percentil
(só min-max) × sem recorte da região não-nula. As features de cada configuração ficam em cache
(`features_ablacao_*.csv`), então só a primeira execução lê DICOM.

In [ ]:
best_model_obj, best_param_grid = MODEL_GRID[BEST_KEY[1]]

ablation_configs = {
    "pipeline_completo (normalize=True, crop=True)": {"normalize": True, "crop": True},
    "sem_normalizacao_percentil (normalize=False, crop=True)": {"normalize": False, "crop": True},
    "sem_recorte (normalize=True, crop=False)": {"normalize": True, "crop": False},
}

ablation_rows = []
for config_name, kwargs in ablation_configs.items():
    if kwargs == {"normalize": True, "crop": True}:
        feat_df = all_features_df          # já calculado (mesmo pipeline)
    else:
        slug = "norm" if kwargs["normalize"] else "nonorm"
        slug += "_crop" if kwargs["crop"] else "_nocrop"
        feat_df = load_or_build(f"features_ablacao_{slug}.csv",
                                lambda kw=kwargs: build_central_features(**kw), PATIENT_IDS)
    if feat_df is None:
        print(f"[pulado] {config_name}")
        continue
    cols = [c for c in best_desc_cols if c in feat_df.columns]
    merged = df.merge(feat_df[["patient_id_str"] + cols], on="patient_id_str", how="inner")
    f_df, o_df = evaluate_nested_cv(merged, cols, best_model_obj, best_param_grid)
    s = summarise(f_df, o_df)
    ablation_rows.append({"configuracao": config_name, "auc_mean": s["auc_mean"], "auc_std": s["auc_std"],
                          "auc_pooled": s["auc_pooled"], "auc_pr_mean": s["auc_pr_mean"],
                          "balanced_accuracy_mean": s["balanced_accuracy_mean"], "accuracy_mean": s["accuracy_mean"]})
    print(f"{config_name} -> AUC {fmt(s['auc_mean'], s['auc_std'])} | agrupado {s['auc_pooled']:.3f}")

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(os.path.join(OUT_DIR, "estudo_ablacao_preprocessamento.csv"), index=False)
ablation_df

**Como interpretar:** as diferenças entre configurações (≈0,05–0,10 de AUC) são menores ou comparáveis ao
desvio entre dobras — registre a contribuição de cada etapa como *indício*, não como efeito comprovado.
Mecanismo plausível: sem o clipping por percentil, poucos pixels muito brilhantes comprimem a escala
e a quantização em 32 níveis do GLCM perde detalhe.

## 13. Ablação da estratégia de agregação de cortes (§4.2 do TP1)

O §4.2 pede que a estratégia de agregação seja uma decisão explícita. Comparamos, com a **mesma** melhor combinação e a **mesma** partição:

| Estratégia | Descrição |
|---|---|
| `central (K=1)` | corte do meio (usado na grade principal) |
| `pool K=3 / 5 / 9` | média das features dos K cortes centrais do paciente |
| `pool K=5 média+dp` | média **e** desvio-padrão entre os 5 cortes centrais |

Cada corte é pré-processado e descrito como no pipeline principal. Os cortes centrais são extraídos **uma vez**
(9 por paciente) e salvos em `features_multicorte_g3.csv`.
*Ressalva:* é uma ablação exploratória sobre as mesmas dobras; a grade principal permanece com o corte central
(experimento congelado), salvo decisão explícita do grupo de adotar o pooling.

In [ ]:
def build_multislice_features():
    rows, failed = [], []
    for pid in PATIENT_IDS:
        try:
            imgs = get_slices_by_offset(pid, FEATURE_MODALITY, MULTISLICE_OFFSETS)
            for off, img in imgs.items():
                feats = {"patient_id_str": pid, "offset": off}
                feats.update(extract_all_families(img))
                rows.append(feats)
        except Exception as e:
            failed.append((pid, str(e)))
    if failed:
        print("Falhas:", failed)
    return pd.DataFrame(rows)

agg_df = None
if RUN_AGG_ABLATION:
    agg_df = load_or_build("features_multicorte_g3.csv", build_multislice_features, PATIENT_IDS)

def pool_features(agg, cols, K, with_std=False):
    half = K // 2
    sub = agg[agg["offset"].between(-half, half)]
    g = sub.groupby("patient_id_str")[cols]
    out = g.mean()
    if with_std:
        sd = g.std(ddof=0).add_suffix("_dp")
        out = out.join(sd)
    return out.reset_index()

if agg_df is not None:
    agg_rows = []
    variantes = [("central (K=1)", 1, False), ("pool K=3 (média)", 3, False), ("pool K=5 (média)", 5, False),
                 ("pool K=9 (média)", 9, False), ("pool K=5 (média+dp)", 5, True)]
    for nome, K, with_std in variantes:
        pooled = pool_features(agg_df, best_desc_cols, K, with_std)
        cols = [c for c in pooled.columns if c != "patient_id_str"]
        merged = df.merge(pooled, on="patient_id_str", how="inner")
        f_df, o_df = evaluate_nested_cv(merged, cols, best_model_obj, best_param_grid)
        s = summarise(f_df, o_df)
        agg_rows.append({"estrategia": nome, "n_features": len(cols), "auc_mean": s["auc_mean"],
                         "auc_std": s["auc_std"], "auc_pooled": s["auc_pooled"], "auc_pr_mean": s["auc_pr_mean"],
                         "balanced_accuracy_mean": s["balanced_accuracy_mean"]})
        print(f"{nome:22s} -> AUC {fmt(s['auc_mean'], s['auc_std'])} | agrupado {s['auc_pooled']:.3f}")
    agg_ablation_df = pd.DataFrame(agg_rows)
    agg_ablation_df.to_csv(os.path.join(OUT_DIR, "estudo_ablacao_agregacao_cortes.csv"), index=False)
    display(agg_ablation_df) if "display" in globals() else print(agg_ablation_df)
else:
    print("Ablação de agregação não executada (RUN_AGG_ABLATION=False ou sem DICOM/cache).")

## 14. Redução de dimensionalidade no conjunto combinado (§4.3 do TP1)

O conjunto Combinado tem 36 features para ~48 pacientes de treino por dobra, e a regressão logística piorou ao juntá-las.
Testamos `SelectKBest` (ANOVA F) **dentro do `Pipeline`** (ajustado só no treino de cada dobra), com `k` e `C` no grid interno.
Análise exploratória: **não** entra na seleção entre as 12 combinações da Seção 8.

In [ ]:
merged_comb = df.merge(all_features_df[["patient_id_str"] + COMBINED_COLS], on="patient_id_str", how="inner")
f_sel, o_sel = evaluate_nested_cv(
    merged_comb, COMBINED_COLS,
    LogisticRegression(max_iter=1000, random_state=SEED),
    {"sel__k": [5, 10, 20], "clf__C": [0.1, 1.0, 10.0]},
    selector=SelectKBest(f_classif))
s_k = summarise(f_sel, o_sel)
print(f"Combinado + SelectKBest + LogReg -> AUC {fmt(s_k['auc_mean'], s_k['auc_std'])} | "
      f"agrupado {s_k['auc_pooled']:.3f} | AUC-PR {s_k['auc_pr_mean']:.3f}")
print(f_sel[["fold", "auc", "melhores_hiperparametros"]].to_string(index=False))
pd.DataFrame([{"modelo": "Combinado + SelectKBest + LogReg", **s_k}]).to_csv(
    os.path.join(OUT_DIR, "extra_selecao_features_combinado.csv"), index=False)

## 15. Preparação para a análise de erro (Marco 4)

1. **Erros com ID do paciente** (`erros_melhor_combinacao.csv`), ordenados pela confiança do modelo.
2. **Plano de aquisição da T1wCE** de cada paciente (lido do cabeçalho DICOM, sem carregar pixels). O plano varia entre pacientes
   (coronal/axial/sagital); se ele se associar ao rótulo ou aos erros, as features (sobretudo forma) podem estar capturando a
   *aquisição*, e não a biologia do tumor — um possível confundidor a discutir.
3. Figura com os erros mais confiantes (falsos positivos e falsos negativos).

In [ ]:
best_oof["erro"] = (best_oof["pred"] != best_oof["y_true"]).astype(int)
best_oof["confianca_no_erro"] = np.where(best_oof["y_true"] == 0, best_oof["proba"], 1 - best_oof["proba"])
erros = best_oof[best_oof["erro"] == 1].sort_values("confianca_no_erro", ascending=False)
erros.assign(tipo=np.where(erros["y_true"] == 0, "falso_positivo", "falso_negativo")).to_csv(
    os.path.join(OUT_DIR, "erros_melhor_combinacao.csv"), index=False)
best_oof.to_csv(os.path.join(OUT_DIR, "predicoes_oof_melhor_combinacao.csv"), index=False)
print(f"Erros: {len(erros)} de {len(best_oof)} "
      f"(FP = {(erros['y_true'] == 0).sum()}, FN = {(erros['y_true'] == 1).sum()})")
erros.head(8)

In [ ]:
def classify_plane(iop):
    if iop is None or len(iop) != 6:
        return "desconhecido"
    normal = np.cross(np.array(iop[0:3], dtype=float), np.array(iop[3:6], dtype=float))
    return {0: "sagital", 1: "coronal", 2: "axial"}[int(np.argmax(np.abs(normal)))]

def build_planes():
    rows = []
    for pid in PATIENT_IDS:
        files = sorted(glob.glob(os.path.join(TRAIN_DIR, pid, FEATURE_MODALITY, "*.dcm")))
        try:
            ds = pydicom.dcmread(files[0], stop_before_pixels=True)
            plane = classify_plane(getattr(ds, "ImageOrientationPatient", None))
        except Exception:
            plane = "desconhecido"
        rows.append({"patient_id_str": pid, "plano_t1wce": plane})
    return pd.DataFrame(rows)

planes_df = load_or_build("plano_t1wce_por_paciente.csv", build_planes, PATIENT_IDS)

if planes_df is not None:
    an = best_oof.merge(planes_df, on="patient_id_str")
    print("Plano da T1wCE × rótulo:")
    ct1 = pd.crosstab(an["plano_t1wce"], an["y_true"])
    print(ct1)
    print("\nPlano da T1wCE × erro do modelo (0 = acerto, 1 = erro):")
    ct2 = pd.crosstab(an["plano_t1wce"], an["erro"])
    print(ct2)
    for nome, ct in [("plano × rótulo", ct1), ("plano × erro", ct2)]:
        if ct.shape[0] > 1 and ct.shape[1] > 1:
            print(f"  qui-quadrado {nome}: p = {chi2_contingency(ct)[1]:.3f} (contagens pequenas — interpretar com cautela)")
    an.to_csv(os.path.join(OUT_DIR, "predicoes_com_plano.csv"), index=False)
else:
    print("Plano de aquisição não disponível (sem cache e sem DICOM).")

In [ ]:
if RUN_ERROR_PLOTS and TRAIN_DIR and os.path.isdir(TRAIN_DIR):
    try:
        fps = erros[erros["y_true"] == 0].head(3)
        fns = erros[erros["y_true"] == 1].head(3)
        casos = [(r, "Falso positivo") for _, r in fps.iterrows()] + [(r, "Falso negativo") for _, r in fns.iterrows()]
        fig, axes = plt.subplots(2, 3, figsize=(11, 7.5))
        for ax, (r, tipo) in zip(axes.ravel(), casos):
            img = get_middle_slice(r["patient_id_str"], FEATURE_MODALITY)
            ax.imshow(img, cmap="gray")
            ax.set_title(f"{tipo} — paciente {r['patient_id_str']}\nreal = {int(r['y_true'])} | P(metilado) = {r['proba']:.2f}",
                         fontsize=9)
            ax.axis("off")
        for ax in axes.ravel()[len(casos):]:
            ax.axis("off")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "exemplos_erros_melhor_combinacao.png"), dpi=150)
        plt.show()
    except Exception as e:
        print("Não foi possível gerar a figura de erros:", e)
else:
    print("Figura de erros não gerada (RUN_ERROR_PLOTS=False ou sem DICOM).")

## 16. Metadados da execução e conferência dos arquivos

In [ ]:
meta = {"seed": SEED, "n_pacientes": int(len(df)), "n_dobras_externas": int(df["fold"].nunique()),
        "dobras_internas": 3, "modalidade": FEATURE_MODALITY, "glcm_distancia": GLCM_DISTANCE,
        "glcm_niveis_cinza": N_GRAY_LEVELS, "n_boot": N_BOOT, "n_perm": N_PERM, "perm_scope": PERM_SCOPE,
        "melhor_combinacao_dobras_externas": f"{BEST_KEY[0]} + {BEST_KEY[1]}",
        "versoes": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                    "scikit-learn": sklearn.__version__, "scikit-image": skimage.__version__,
                    "opencv": cv2.__version__, "pydicom": pydicom.__version__, "scipy": scipy.__version__}}
with open(os.path.join(OUT_DIR, "metadados_execucao.json"), "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta["versoes"], indent=2))

print(f"\nArquivos gerados em '{OUT_DIR}/':")
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

## 17. Notas para o artigo e pendências do Marco 4

**Como reportar (sugestão):**
- Tabela principal: `grade_comparativa_descritor_modelo.csv` (média ± dp) + linha do baseline trivial.
- Desempenho final do procedimento: **seleção aninhada** (Seção 8), não o melhor das 12 combinações.
- Sempre acompanhar o AUC por dobra do **AUC agrupado** e do **IC 95%**; citar o p-valor de permutação.
- Discutir: sinal fraco (Kim et al., 2022), incerteza do rótulo (Brandner et al., 2021; Poon et al., 2021), e ausência de máscara do tumor.

**Pendências:**
- [ ] Análise de erro qualitativa (usar `erros_melhor_combinacao.csv`, `predicoes_com_plano.csv`, `exemplos_erros_*.png`)
- [ ] Decidir e justificar a estratégia final de agregação (Seção 13)
- [ ] Citações metodológicas: Haralick (GLCM), Hu (momentos), Dalal & Triggs (gradiente), Otsu, normalização de RM
- [ ] Redigir Metodologia/Resultados/Conclusão; caber em 4 páginas (template SBC)
- [ ] `requirements.txt` com versões (ver `metadados_execucao.json`), README atualizado, link do repositório no artigo
- [ ] Teste de reprodutibilidade em ambiente limpo, seguindo só o README